# Lab Assignment 4

MST Dependency Parsing and Training

**Total: 20 marks**

## Objective
Implement core components of an MST dependency parser, analyse a Chu-Liu/Edmonds cycle, and perform one structured-perceptron update using gold and predicted dependency trees.

## Submission
- Complete all `TODO` sections and all markdown answers.
- Do not modify the supplied data or expected-test cells.
- Run **Kernel > Restart & Run All** before submitting the completed `.ipynb` file.

## Valid dependency tree conditions
Each non-root word has exactly one incoming head; `ROOT` has none; all words are reachable from `ROOT`; and no directed cycle occurs.

In [9]:
from itertools import product
from collections import defaultdict, Counter
import pandas as pd
import matplotlib.pyplot as plt

## Task 1: Validate Dependency Trees (2 marks)

Implement `is_valid_tree()` to verify whether a set of dependency arcs forms a valid rooted dependency tree.

A valid dependency tree must satisfy all of the following conditions:

1. `ROOT` has no incoming arc.
2. Every non-root word has exactly one incoming head.
3. No word can be its own head (no self-loop).
4. The graph must not contain a directed cycle.
5. Every word must be reachable from `ROOT`.
6. The number of arcs must be exactly `number_of_tokens - 1`.

### Your task

Complete the `is_valid_tree()` function below.

Do not change the test cases.

Your function should return:

- `True` for a valid dependency tree.
- `False` for an invalid dependency structure.

You should ensure that the function correctly detects duplicate heads, missing heads, cycles, self-loops, disconnected components, and arcs entering `ROOT`.

In [10]:
def is_valid_tree(tokens, arcs, root="ROOT"):
    non_root = [w for w in tokens if w != root]

    # A rooted dependency tree has exactly one incoming arc per non-root node.
    if len(arcs) != len(non_root):
        return False

    heads = {}

    for head, dep in arcs:
        # Nodes must belong to the sentence.
        if head not in tokens or dep not in tokens:
            return False

        # ROOT cannot have an incoming dependency arc.
        if dep == root:
            return False

        # No self-loop.
        if head == dep:
            return False

        # Exactly one head per dependent.
        if dep in heads:
            return False

        heads[dep] = head

    # Every non-root token must occur exactly once as a dependent.
    if set(heads) != set(non_root):
        return False

    # Every node must reach ROOT; repeated nodes indicate a cycle.
    for start in non_root:
        current = start
        visited = set()

        while current != root:
            if current in visited:
                return False
            visited.add(current)

            if current not in heads:
                return False

            current = heads[current]

    return True


test_tokens = ["ROOT", "A", "B", "C"]
valid = [("ROOT","A"), ("A","B"), ("A","C")]
cycle = [("ROOT","A"), ("B","C"), ("C","B")]
duplicate = [("ROOT","A"), ("A","B"), ("C","B")]
print(is_valid_tree(test_tokens, valid))      # Expected: True
print(is_valid_tree(test_tokens, cycle))      # Expected: False
print(is_valid_tree(test_tokens, duplicate))  # Expected: False


True
False
False


## Task 2: Exhaustive MST Reference Parser (3 marks)

Implement an exhaustive reference parser for a small dependency graph.

For every non-root word:

1. Enumerate all possible incoming heads.
2. Construct candidate dependency trees.
3. Use `is_valid_tree()` from Task 1 to reject invalid structures.
4. Compute the total score of every valid tree.
5. Return the highest-scoring tree and its score.

The score of a tree is:

$$
Score(G)=\sum_{(h,d)\in G} Score(h\rightarrow d)
$$

where:

- $h$ = head
- $d$ = dependent
- $G$ = dependency tree

### Your task

Complete:

```python
exhaustive_mst(tokens, score)

In [11]:
tokens = ["ROOT", "Researchers", "analyze", "data", "carefully"]

score = {
    ("ROOT","Researchers"):1,
    ("ROOT","analyze"):7,
    ("ROOT","data"):1,
    ("ROOT","carefully"):1,

    ("Researchers","analyze"):2,
    ("Researchers","data"):2,
    ("Researchers","carefully"):2,

    ("analyze","Researchers"):6,
    ("analyze","data"):7,
    ("analyze","carefully"):6,

    ("data","Researchers"):2,
    ("data","analyze"):3,
    ("data","carefully"):3,

    ("carefully","Researchers"):2,
    ("carefully","analyze"):3,
    ("carefully","data"):4,
}

def exhaustive_mst(tokens, score, root="ROOT"):
    non_root = [w for w in tokens if w != root]
    choices = [[h for h in tokens if h != dep] for dep in non_root]

    best_score = float("-inf")
    best_arcs = None

    for heads in product(*choices):
        candidate = list(zip(heads, non_root))
        if not is_valid_tree(tokens, candidate, root):
            continue

        total = sum(score[(h, d)] for h, d in candidate)
        if total > best_score:
            best_score = total
            best_arcs = candidate

    return best_score, best_arcs

best_score, best_arcs = exhaustive_mst(tokens, score)

print("Best score:", best_score)
print("Best arcs:", best_arcs)


Best score: 26
Best arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]


## Task 3: Chu-Liu/Edmonds — Best Incoming Edges and Cycle Detection (3 marks)

The Chu-Liu/Edmonds algorithm starts by selecting the highest-scoring incoming edge for every non-root node.

For each node $v$:

$$
parent(v)
=
\arg\max_{u\neq v} Score(u\rightarrow v)
$$

If the selected edges form a valid tree, the algorithm is finished.

However, if the selected edges contain a directed cycle, the cycle must be detected and contracted before continuing.

### Your task

Complete the following two functions:

1. `best_incoming()`
2. `find_cycle()`

`best_incoming()` should select the highest-scoring incoming edge for every non-root node.

`find_cycle()` should:

- return one directed cycle as a list of nodes if a cycle exists;
- return `None` if no cycle exists.

### Questions

After running the code:

1. What edges were selected by the greedy incoming-edge step?
2. Which nodes form the cycle?
3. Why is the greedy result not a valid dependency tree?
4. Why can we not simply accept the greedy result as the MST?

In [12]:
cycle_tokens = ["ROOT", "A", "B", "C"]

cycle_score = {
    ("ROOT","A"):5,
    ("ROOT","B"):4,
    ("ROOT","C"):3,

    ("A","B"):10,
    ("B","A"):10,

    ("A","C"):7,
    ("B","C"):8,

    ("C","A"):2,
    ("C","B"):2,
}

def best_incoming(tokens, score, root="ROOT"):
    arcs = []
    for dep in tokens:
        if dep == root:
            continue
        candidates = [
            (head, dep)
            for head in tokens
            if head != dep and dep != root and (head, dep) in score
        ]
        if not candidates:
            raise ValueError(f"No incoming edge available for {dep}")
        arcs.append(max(candidates, key=lambda arc: score[arc]))
    return arcs

def find_cycle(arcs):
    parent = {dep: head for head, dep in arcs}

    for start in parent:
        current = start
        path = []
        position = {}

        while current in parent:
            if current in position:
                return path[position[current]:]
            position[current] = len(path)
            path.append(current)
            current = parent[current]

    return None

greedy_arcs = best_incoming(cycle_tokens, cycle_score)

print("Greedy arcs:", greedy_arcs)
print("Cycle:", find_cycle(greedy_arcs))


Greedy arcs: [('B', 'A'), ('A', 'B'), ('B', 'C')]
Cycle: ['A', 'B']


**Task 3 explanation:**

1. **Greedy incoming edges:** `ROOT -> A`, `A -> B`, `B -> A`, and `B -> C`.

2. **Cycle:** The nodes `A` and `B` form the directed cycle `A -> B -> A`.

3. **Why is it invalid?** A dependency tree cannot contain a directed cycle. Following heads from `A` gives `A -> B -> A`, so neither node can reach `ROOT`.

4. **Why can we not accept the greedy result as the MST?** Choosing the highest-scoring incoming edge independently for every node does not enforce the global tree constraint. The locally best edges can contain a cycle, so Chu-Liu/Edmonds must contract the cycle, adjust the relevant edge weights, solve the contracted graph, and then expand the cycle.


## Task 4: Chu-Liu/Edmonds Cycle Contraction (3 marks)

Suppose the greedy step produces the cycle:

$$
A\rightarrow B
$$

and

$$
B\rightarrow A
$$

with:

$$
Score(A\rightarrow B)=10
$$

$$
Score(B\rightarrow A)=10
$$

The total score of the selected cycle is:

$$
CycleScore=10+10=20
$$

When an external edge enters a node inside the cycle, the cycle is contracted into a single **supernode**.

For an external edge $h\rightarrow d$ entering the cycle, calculate the adjusted score as:

$$
\boxed{
AdjustedScore(h\rightarrow d)
=
Score(h\rightarrow d)
+
CycleScore
-
Score(old\_incoming\_edge(d))
}
$$

### Example

For:

$$
ROOT\rightarrow A
$$

we have:

$$
Score(ROOT\rightarrow A)=5
$$

The selected edge entering `A` inside the cycle is:

$$
B\rightarrow A
$$

with score:

$$
Score(B\rightarrow A)=10
$$

Therefore:

$$
AdjustedScore(ROOT\rightarrow A)
=
5+20-10
=
15
$$

### Your task

Complete `entering_cycle_table()`.

Create a table for:

- `ROOT -> A`
- `ROOT -> B`

The table must contain:

- candidate edge
- original score
- selected cycle edge being removed
- removed edge score
- total cycle score
- adjusted score

### Interpretation question

Which candidate edge should be selected to enter the contracted cycle?

Explain which original cycle edge will be removed when the contracted tree is expanded back to the original graph.

In [13]:
def entering_cycle_table(score, cycle, selected_cycle_arcs):
    cycle_set = set(cycle)
    cycle_total = sum(score[arc] for arc in selected_cycle_arcs)
    rows = []

    # The assignment asks for ROOT -> A and ROOT -> B.
    candidates = [("ROOT", node) for node in cycle if ("ROOT", node) in score]

    for head, dep in candidates:
        original_score = score[(head, dep)]
        removed_arc = next(
            arc for arc in selected_cycle_arcs if arc[1] == dep
        )
        removed_score = score[removed_arc]
        adjusted_score = original_score + cycle_total - removed_score

        rows.append({
            "outside_head": head,
            "cycle_dependent": dep,
            "original_score": original_score,
            "removed_cycle_arc": removed_arc,
            "removed_score": removed_score,
            "cycle_total": cycle_total,
            "adjusted_score": adjusted_score
        })

    return pd.DataFrame(rows)

selected_cycle_arcs = [
    ("B","A"),
    ("A","B")
]

cycle_table = entering_cycle_table(
    cycle_score,
    ["A", "B"],
    selected_cycle_arcs
)
cycle_table


,outside_head,cycle_dependent,original_score,removed_cycle_arc,removed_score,cycle_total,adjusted_score
0,ROOT,A,5,"(B, A)",10,20,15
1,ROOT,B,4,"(A, B)",10,20,14


### Task 4 Interpretation

**Which candidate edge should be selected to enter the contracted cycle?**

The edge `ROOT -> A` should be selected. According to the `cycle_table`, the adjusted score for `ROOT -> A` is **15**, which is higher than the adjusted score for `ROOT -> B` (**14**).

**Which original cycle edge will be removed when the contracted tree is expanded back?**

When `ROOT -> A` is selected as the entry point into the cycle, the original cycle edge that entered node `A` must be removed to ensure every node has exactly one head. In the cycle `A <-> B`, the edge entering `A` is `B -> A`. Therefore, `B -> A` is removed, and the remaining cycle edge `A -> B` is retained. The final arcs for this subgraph would be `ROOT -> A` and `A -> B`.

In [14]:
def select_best_incoming(nodes, edges, root="ROOT"):
    """Select the highest-scoring incoming edge for every non-root node."""
    selected = []

    for dep in nodes:
        if dep == root:
            continue

        candidates = [
            (head, dep)
            for head in nodes
            if head != dep and dep != root and (head, dep) in edges
        ]

        if not candidates:
            raise ValueError(f"No incoming edge for {dep}")

        selected.append(max(candidates, key=lambda arc: edges[arc]))

    return selected


def detect_cycle(arcs, root="ROOT"):
    """Return one directed cycle from arcs, or None."""
    parent = {dep: head for head, dep in arcs}

    for start in parent:
        current = start
        path = []
        position = {}

        while current in parent:
            if current in position:
                return path[position[current]:]

            position[current] = len(path)
            path.append(current)
            current = parent[current]

            if current == root:
                break

    return None


def contract_cycle(nodes, edges, selected_arcs, cycle, root="ROOT"):
    """Contract one detected cycle and retain enough metadata to expand it."""
    cycle_set = set(cycle)
    supernode = "__CYCLE__"

    contracted_nodes = [n for n in nodes if n not in cycle_set] + [supernode]
    contracted_edges = {}

    incoming_map = {}
    outgoing_map = {}
    ordinary_map = {}

    cycle_arcs = [
        arc for arc in selected_arcs
        if arc[0] in cycle_set and arc[1] in cycle_set
    ]
    cycle_total = sum(edges[arc] for arc in cycle_arcs)

    selected_incoming = {
        dep: (head, dep)
        for head, dep in selected_arcs
        if dep in cycle_set
    }

    for (head, dep), weight in edges.items():
        if head == dep:
            continue

        # Internal cycle edges disappear after contraction.
        if head in cycle_set and dep in cycle_set:
            continue

        # Edge entering the cycle.
        if head not in cycle_set and dep in cycle_set:
            key = (head, supernode)
            old_arc = selected_incoming[dep]
            adjusted = weight + cycle_total - edges[old_arc]

            # Keep the best parallel contracted edge.
            if key not in contracted_edges or adjusted > contracted_edges[key]:
                contracted_edges[key] = adjusted
                incoming_map[key] = (head, dep)

        # Edge leaving the cycle.
        elif head in cycle_set and dep not in cycle_set:
            key = (supernode, dep)
            if key not in contracted_edges or weight > contracted_edges[key]:
                contracted_edges[key] = weight
                outgoing_map[key] = (head, dep)

        # Edge outside the cycle.
        else:
            key = (head, dep)
            if key not in contracted_edges or weight > contracted_edges[key]:
                contracted_edges[key] = weight
                ordinary_map[key] = (head, dep)

    mapping_information = {
        "supernode": supernode,
        "cycle": cycle_set,
        "incoming_map": incoming_map,
        "outgoing_map": outgoing_map,
        "ordinary_map": ordinary_map,
        "selected_incoming": selected_incoming,
        "cycle_arcs": cycle_arcs,
        "cycle_total": cycle_total
    }

    return contracted_nodes, contracted_edges, mapping_information


def expand_cycle(
    selected_contracted_arcs,
    cycle,
    mapping_information,
    original_selected_arcs
):
    """Expand the recursive solution and remove the replaced cycle edge."""
    supernode = mapping_information["supernode"]
    incoming_map = mapping_information["incoming_map"]
    outgoing_map = mapping_information["outgoing_map"]

    cycle_set = set(cycle)

    # Start with the selected cycle edges. Exactly one of these will be removed.
    final_arcs = list(mapping_information["cycle_arcs"] )

    entering_original = None
    entered_node = None

    for arc in selected_contracted_arcs:
        head, dep = arc

        if dep == supernode:
            entering_original = incoming_map[arc]
            entered_node = entering_original[1]

        elif head == supernode:
            final_arcs.append(outgoing_map[arc])

        else:
            # This edge never touches the contracted cycle.
            final_arcs.append(arc)

    if entering_original is None:
        raise ValueError("A valid contracted arborescence must enter the cycle")

    # Replace the selected cycle node's old incoming edge with the external edge.
    final_arcs = [
        arc for arc in final_arcs
        if not (arc[1] == entered_node and arc[0] in cycle_set)
    ]
    final_arcs.append(entering_original)

    return final_arcs


def chu_liu_edmonds(tokens, score, root="ROOT"):
    """Maximum-scoring rooted directed spanning tree using Chu-Liu/Edmonds."""

    nodes = list(tokens)

    # Step 1: best incoming edge for every non-root node.
    selected = select_best_incoming(nodes, score, root)

    # Step 2: detect a directed cycle.
    cycle = detect_cycle(selected, root)

    # No cycle means the selected edges already form the maximum arborescence.
    if cycle is None:
        total = sum(score[arc] for arc in selected)
        return total, selected

    # Step 3/4: contract the cycle and adjust incoming weights.
    contracted_nodes, contracted_edges, mapping = contract_cycle(
        nodes, score, selected, cycle, root
    )

    # Step 5: recursively solve the contracted graph.
    _, contracted_solution = chu_liu_edmonds(
        contracted_nodes, contracted_edges, root
    )

    # Step 6: expand the cycle back to original nodes.
    final_arcs = expand_cycle(
        contracted_solution,
        cycle,
        mapping,
        selected
    )

    total = sum(score[arc] for arc in final_arcs)
    return total, final_arcs


# Test on the supplied graph
cle_score, cle_arcs = chu_liu_edmonds(tokens, score)

print("Chu-Liu/Edmonds score:", cle_score)
print("Chu-Liu/Edmonds arcs:", cle_arcs)
print("Matches exhaustive MST score?", cle_score == best_score)
print("Valid final tree?", is_valid_tree(tokens, cle_arcs))


Chu-Liu/Edmonds score: 26
Chu-Liu/Edmonds arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]
Matches exhaustive MST score? True
Valid final tree? True


## Task 6: Inference-Based Learning with Structured Perceptron (5 marks)

In this task, you will train the MST dependency parser using **inference-based structured perceptron learning**.

The parser uses the current weight vector to score every possible dependency tree and selects the highest-scoring tree:

$$
\boxed{
G_{pred}
=
\arg\max_{G\in T(x)}
\mathbf{w}\cdot\Phi(x,G)
}
$$

where:

- $x$ = input sentence
- $T(x)$ = set of valid dependency trees for the sentence
- $\mathbf{w}$ = current feature-weight vector
- $\Phi(x,G)$ = feature vector of dependency tree $G$

The predicted tree is then compared with the gold tree.

If:

$$
G_{pred}\neq G_{gold}
$$

update the weights using:

$$
\boxed{
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
}
$$

Equivalently, for each feature:

$$
\boxed{
w_{new}[f]
=
w_{old}[f]
+
\Phi(x,G_{gold})[f]
-
\Phi(x,G_{pred})[f]
}
$$

### Training procedure

Your training loop must perform the following steps:

1. Initialize the feature weights.
2. Take one training sentence and its gold dependency tree.
3. Compute the score of candidate dependency arcs using the current weights.
4. Run your **own Chu-Liu/Edmonds implementation from Task 5** to obtain the predicted tree.
5. Compute the feature vector of the gold tree.
6. Compute the feature vector of the predicted tree.
7. Update the weights using the structured perceptron rule.
8. Repeat for all training examples.
9. Repeat the complete process for multiple epochs.

### Important

The training procedure must call your implementation from **Task 5**.

Do not use:

```python
networkx.maximum_spanning_arborescence()

In [15]:
from collections import Counter

# ---------------------------------------------------------
# Training data
# ---------------------------------------------------------

training_data = [
    {
        "tokens": ["ROOT", "John", "saw", "Mary"],
        "pos": {
            "ROOT": "ROOT",
            "John": "NOUN",
            "saw": "VERB",
            "Mary": "NOUN"
        },
        "gold": [
            ("ROOT", "saw"),
            ("saw", "John"),
            ("saw", "Mary")
        ]
    },

    {
        "tokens": ["ROOT", "Dogs", "chase", "cats"],
        "pos": {
            "ROOT": "ROOT",
            "Dogs": "NOUN",
            "chase": "VERB",
            "cats": "NOUN"
        },
        "gold": [
            ("ROOT", "chase"),
            ("chase", "Dogs"),
            ("chase", "cats")
        ]
    },

    {
        "tokens": ["ROOT", "Alice", "likes", "Bob"],
        "pos": {
            "ROOT": "ROOT",
            "Alice": "NOUN",
            "likes": "VERB",
            "Bob": "NOUN"
        },
        "gold": [
            ("ROOT", "likes"),
            ("likes", "Alice"),
            ("likes", "Bob")
        ]
    }
]


# ---------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------

def arc_features(tokens, pos, head, dep):
    """Return the seven required feature templates for one arc."""
    head_i = tokens.index(head)
    dep_i = tokens.index(dep)
    direction = "RIGHT" if head_i < dep_i else "LEFT"
    distance = abs(head_i - dep_i)

    return [
        f"HEAD_WORD={head}",
        f"DEP_WORD={dep}",
        f"HEAD_POS={pos[head]}",
        f"DEP_POS={pos[dep]}",
        f"DIRECTION={direction}",
        f"DISTANCE={distance}",
        f"WORD_PAIR={head}->{dep}"
    ]


def tree_feature_vector(tokens, pos, tree):
    features = Counter()
    for head, dep in tree:
        features.update(arc_features(tokens, pos, head, dep))
    return features


# ---------------------------------------------------------
# Arc and tree scoring
# ---------------------------------------------------------

def score_arc(tokens, pos, head, dep, weights):
    return sum(weights[f] for f in arc_features(tokens, pos, head, dep))


def build_score_graph(tokens, pos, weights):
    graph = {}
    for head in tokens:
        for dep in tokens:
            if head == dep or dep == "ROOT":
                continue
            graph[(head, dep)] = score_arc(
                tokens, pos, head, dep, weights
            )
    return graph


def score_tree(tokens, pos, tree, weights):
    return sum(
        score_arc(tokens, pos, head, dep, weights)
        for head, dep in tree
    )


# ---------------------------------------------------------
# Structured perceptron update
# ---------------------------------------------------------

def perceptron_update(
    weights,
    gold_features,
    predicted_features,
    learning_rate=1.0
):
    updated = Counter(weights)

    for feature, count in gold_features.items():
        updated[feature] += learning_rate * count

    for feature, count in predicted_features.items():
        updated[feature] -= learning_rate * count

    return updated


# ---------------------------------------------------------
# Inference-based training
# ---------------------------------------------------------

def train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
):
    weights = Counter()
    history = []

    for epoch in range(num_epochs):
        mistakes = 0

        for example in training_data:
            tokens = example["tokens"]
            pos = example["pos"]
            gold_tree = example["gold"]

            # Build scores using the current weights.
            score_graph = build_score_graph(tokens, pos, weights)

            # Inference MUST use the Task 5 implementation.
            _, predicted_tree = chu_liu_edmonds(
                tokens, score_graph
            )

            if set(predicted_tree) != set(gold_tree):
                mistakes += 1

                gold_features = tree_feature_vector(
                    tokens, pos, gold_tree
                )
                predicted_features = tree_feature_vector(
                    tokens, pos, predicted_tree
                )

                weights = perceptron_update(
                    weights,
                    gold_features,
                    predicted_features,
                    learning_rate
                )

        history.append(mistakes)
        print(f"Epoch {epoch + 1}: {mistakes} mistake(s)")

    return weights, history


In [16]:
# Run the complete inference-based training
final_weights, training_history = train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
)

print("\nTraining history:")
for epoch, mistakes in enumerate(training_history, start=1):
    print(f"Epoch {epoch}: {mistakes} mistake(s)")

print("\nLearned weights:")
for feature, weight in sorted(final_weights.items()):
    print(f"{feature}: {weight}")


Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Training history:
Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Learned weights:
DEP_POS=NOUN: 0.0
DEP_POS=VERB: 0.0
DEP_WORD=John: 0.0
DEP_WORD=Mary: 0.0
DEP_WORD=saw: 0.0
DIRECTION=LEFT: 1.0
DIRECTION=RIGHT: -1.0
DISTANCE=1: 1.0
DISTANCE=2: 0.0
DISTANCE=3: -1.0
HEAD_POS=ROOT: -2.0
HEAD_POS=VERB: 2.0
HEAD_WORD=ROOT: -2.0
HEAD_WORD=saw: 2.0
WORD_PAIR=ROOT->John: -1.0
WORD_PAIR=ROOT->Mary: -1.0
WORD_PAIR=ROOT->saw: 0.0
WORD_PAIR=saw->John: 1.0
WORD_PAIR=saw->Mary: 1.0


## Task 6: Training Analysis

### Q1. Prediction Errors

The completed training run produced the following number of mistakes:

| Epoch | Number of Mistakes |
|------:|-------------------:|
| 1 | 1 |
| 2 | 0 |
| 3 | 0 |
| 4 | 0 |
| 5 | 0 |

### Q2. Weight Updates

Two features whose weights increased are `HEAD_WORD=saw` and `HEAD_POS=VERB`, both ending at **+2**. They are associated with the gold arcs headed by `saw` and were not present in the incorrect first prediction, so the structured perceptron added their gold feature counts. `WORD_PAIR=saw->John` and `WORD_PAIR=saw->Mary` are other positive examples, each ending at **+1**.

### Q3. Negative Updates

Two features whose weights decreased are `HEAD_WORD=ROOT` and `HEAD_POS=ROOT`, both ending at **-2**. They appeared in the incorrect predicted root arcs but not in the gold arcs, so the perceptron subtracts their predicted feature counts. `WORD_PAIR=ROOT->John` and `WORD_PAIR=ROOT->Mary` are also negative, ending at **-1** each.

### Q4. Effect of the Perceptron Update

The update

$$
\mathbf{w}_{new}=\mathbf{w}_{old}+\Phi(x,G_{gold})-\Phi(x,G_{pred})
$$

adds weight to features supporting the gold tree and subtracts weight from features supporting the incorrect prediction. Because the score of each candidate tree is the sum of the weights of its arc features, the gold tree's score becomes relatively larger and the incorrect tree's score becomes relatively smaller during subsequent inference.

### Q5. Training Behaviour

Yes. The number of mistakes falls from **1 in Epoch 1 to 0 in Epoch 2**, and remains at 0 through Epoch 5. The first all-zero-weight inference selects a valid but incorrect tree because of the deterministic tie-breaking order. The perceptron update then strongly favors the gold structure using features such as the `saw` head word/POS and penalizes the incorrect ROOT-headed alternatives. The next epochs therefore recover the gold trees for all three supplied training examples.


## Submission Notes

This notebook contains complete solutions for all six tasks and the Task 6 analysis questions. The Chu-Liu/Edmonds implementation is written from scratch and does not use NetworkX's maximum-spanning-arborescence implementation.

The supplied tutorial emphasizes the same workflow: validate dependency trees, formulate parsing as maximum-scoring directed spanning-tree inference, understand greedy cycles, contract cycles with adjusted weights, recursively solve the contracted graph, expand it, and use that inference procedure inside structured-perceptron training.
